# Math Problem Solver

**Model:** Qwen2.5-14B-Instruct  
**Dataset:** nvidia/OpenMathReasoning (TIR split)  
**Goal:** Solve competition math problems  
**Method:** Best-of-N generation → Verification → Beam search

## Phases
1. Setup & model loading
2. Load OpenMathReasoning TIR dataset
3. Verification engine (code exec + answer extraction)
4. Best-of-N solver with majority vote + benchmark
5. Beam search with verification pruning + benchmark

---
## 1. Setup

In [1]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"

!pip install -q vllm transformers>=4.44.0 datasets sympy pandas tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.33.5 which is incompatible.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.9.2 which is incompatible.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.2.19 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
cudf-cu12 25.6.0 requires cuda-python<13.0a0,>=12.6.2, but you have cuda-python 13.1.1 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU — change runtime to GPU")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:  {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")

# --- Config ---
MODEL_ID = "qingy2024/Qwen2.5-Math-14B-Instruct-Preview"
MAX_TOKENS = 4096          # increased from 2048 — model needs room for code + verification
SOLUTIONS_PER_PROBLEM = 16

os.makedirs("results", exist_ok=True)
print(f"Model: {MODEL_ID}")
print(f"Max tokens per solution: {MAX_TOKENS}")

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

llm = LLM(
    model=MODEL_ID,
    trust_remote_code=True,
    dtype="half",
    max_model_len=MAX_TOKENS + 1024,   # prompt headroom
    gpu_memory_utilization=0.92,
)

print(f"Model loaded: {MODEL_ID}")
print(f"max_model_len: {MAX_TOKENS + 1024}")

---
## 2. Verification Engine

Three signals:
1. **Answer extraction** — pull answer from `\boxed{}`
2. **Code execution** — run any python blocks, check they don't error
3. **Answer validation** — numeric/symbolic comparison

In [ ]:
import re
import subprocess
import sys
import sympy


# ---------------------------------------------------------------------------
# Answer extraction
# ---------------------------------------------------------------------------

def extract_answer(text):
    """Extract final answer from \\boxed{}.  Falls back to 'the answer is X'."""
    matches = []
    for m in re.finditer(r'\\boxed\{', text):
        start = m.end()
        depth = 1
        i = start
        while i < len(text) and depth > 0:
            if text[i] == '{':
                depth += 1
            elif text[i] == '}':
                depth -= 1
            i += 1
        if depth == 0:
            matches.append(text[start:i - 1])
    if matches:
        return matches[-1].strip()
    m = re.search(r'answer\s+is\s*[:\s]*\$?([^\n$]+)', text, re.I)
    if m:
        return m.group(1).strip()
    return None


# ---------------------------------------------------------------------------
# Normalization helpers
# ---------------------------------------------------------------------------

def _strip_outer_parens(s):
    """Strip balanced outer parens — but only when the inner string has no
    top-level comma (so we never collapse tuples like (2,7) to 2,7)."""
    while len(s) >= 2 and s[0] == '(' and s[-1] == ')':
        inner = s[1:-1]
        depth, ok, top_comma = 0, True, False
        for ch in inner:
            if ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
                if depth < 0:
                    ok = False
                    break
            elif ch == ',' and depth == 0:
                top_comma = True
        if ok and depth == 0 and not top_comma:
            s = inner
        else:
            break
    return s


def _preprocess_latex(ans):
    """Convert LaTeX markup into a cleaner string before normalization."""
    # dfrac → frac (must precede backslash-strip)
    ans = ans.replace('\\dfrac', '\\frac')

    # Named constants
    ans = re.sub(r'\\pi(?![a-zA-Z])', 'pi', ans)
    ans = re.sub(r'\\infty', 'oo', ans)
    ans = re.sub(r'\\cdot', '*', ans)
    ans = re.sub(r'\\times', '*', ans)
    ans = re.sub(r'\\circ\b', '', ans)          # degree symbol (cosmetic)
    ans = re.sub(r'\\approx', '~', ans)

    # \left( → (   \right) → )   etc.
    ans = re.sub(r'\\(?:left|right)\s*\(', '(', ans)
    ans = re.sub(r'\\(?:left|right)\s*\)', ')', ans)
    ans = re.sub(r'\\(?:left|right)\s*\[', '[', ans)
    ans = re.sub(r'\\(?:left|right)\s*\]', ']', ans)
    ans = re.sub(r'\\(?:left|right)\.', '', ans)

    # \frac{num}{den} → ((num)/(den))  — up to 6 nesting levels
    for _ in range(6):
        new = re.sub(r'\\frac\{([^{}]*)\}\{([^{}]*)\}', r'((\1)/(\2))', ans)
        if new == ans:
            break
        ans = new

    # \sqrt{expr} → sqrt(expr)
    for _ in range(4):
        new = re.sub(r'\\sqrt\{([^{}]*)\}', r'sqrt(\1)', ans)
        if new == ans:
            break
        ans = new

    # Text wrappers
    ans = re.sub(r'\\(?:text|mathrm|mathbf|textbf|mbox)\{([^}]*)\}', r'\1', ans)

    # Strip remaining LaTeX
    ans = re.sub(r'[\\${}]', '', ans)
    ans = re.sub(r'\s+', '', ans)

    # Strip outer parens (single-value, not tuples)
    ans = _strip_outer_parens(ans)

    return ans


def _try_sympy(s):
    """Try to parse s as a SymPy expression.  Returns (expr, success)."""
    try:
        # Convert ^ to ** for exponentiation (LaTeX remnant)
        s2 = re.sub(r'\^(-?\w+(?:\.\w+)?)', r'**(\1)', s)
        s2 = re.sub(r'\^\(([^)]*)\)', r'**(\1)', s2)
        val = sympy.sympify(s2)
        return val, True
    except Exception:
        return None, False


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def normalize_answer(answer):
    """Normalize a raw (possibly LaTeX) answer string for comparison."""
    if not answer:
        return ""

    ans = _preprocess_latex(answer.strip())

    # Try plain float first
    try:
        val = float(ans)
        if val == int(val):
            return str(int(val))
        return f"{val:.6f}".rstrip('0').rstrip('.')
    except (ValueError, TypeError):
        pass

    # Try SymPy numeric evaluation
    val, ok = _try_sympy(ans)
    if ok and val is not None and getattr(val, 'is_number', False):
        try:
            f = float(val)
            if abs(f - round(f)) < 1e-9:
                return str(int(round(f)))
            return f"{f:.6f}".rstrip('0').rstrip('.')
        except Exception:
            pass

    return ans.lower()


def _split_toplevel(s):
    """Split s at top-level commas (respects parentheses depth).
    Also converts ' and ' → ',' before splitting."""
    s = re.sub(r'\s*\band\b\s*', ',', s, flags=re.I)
    parts, depth, current = [], 0, ''
    for ch in s:
        if ch in '([':
            depth += 1
            current += ch
        elif ch in ')]':
            depth -= 1
            current += ch
        elif ch == ',' and depth == 0:
            p = _strip_outer_parens(current.strip())
            if p:
                parts.append(p)
            current = ''
        else:
            current += ch
    p = _strip_outer_parens(current.strip())
    if p:
        parts.append(p)
    return parts


def answers_match(a, b):
    """Return True if two *normalised* answers are mathematically equivalent."""
    if not a or not b:
        return False
    if a == b:
        return True

    # Numeric comparison
    try:
        return abs(float(a) - float(b)) < 1e-4
    except (ValueError, TypeError):
        pass

    # SymPy symbolic comparison (handles (π-2)/2 == π/2-1, etc.)
    va, oka = _try_sympy(a)
    vb, okb = _try_sympy(b)
    if oka and okb and va is not None and vb is not None:
        try:
            diff = sympy.simplify(va - vb)
            if diff == 0:
                return True
            if getattr(diff, 'is_number', False) and abs(float(diff)) < 1e-6:
                return True
        except Exception:
            pass

    # Multi-value / set comparison  e.g.  "(2,7),(3,17)"  vs  "((2,7)) and ((3,17))"
    parts_a = _split_toplevel(a)
    parts_b = _split_toplevel(b)
    if len(parts_a) > 1 and len(parts_a) == len(parts_b):
        if sorted(parts_a) == sorted(parts_b):
            return True
        # element-wise after individual normalisation
        if all(answers_match(x, y)
               for x, y in zip(sorted(parts_a), sorted(parts_b))):
            return True

    return False


# ---------------------------------------------------------------------------
# Code execution & scoring
# ---------------------------------------------------------------------------

def execute_code_blocks(text, timeout=10):
    """Run python code blocks from solution. Returns (passed, failed, output)."""
    blocks = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
    if not blocks:
        return 0, 0, ""
    combined = "\n".join(b.strip() for b in blocks)
    try:
        result = subprocess.run(
            [sys.executable, "-c", combined],
            capture_output=True, text=True, timeout=timeout,
        )
        if result.returncode == 0:
            return len(blocks), 0, result.stdout.strip()
        else:
            return 0, len(blocks), result.stderr.strip().split('\n')[-1][:200]
    except subprocess.TimeoutExpired:
        return 0, len(blocks), "timeout"
    except Exception as e:
        return 0, len(blocks), str(e)[:200]


def score_solution(text):
    """Score a solution. Higher = more trustworthy."""
    score = 0.0
    answer = extract_answer(text)
    if answer:
        score += 1.0
    passed, failed, _ = execute_code_blocks(text)
    score += 1.5 * passed    # reward each passing block
    score -= 2.0 * failed    # heavily penalise broken code
    return score, answer


# ---------------------------------------------------------------------------
# Smoke tests
# ---------------------------------------------------------------------------

test = r"""We solve $x^2 = 4$, so $x = 2$.

```python
import sympy
x = sympy.Symbol('x')
print(sympy.solve(x**2 - 4, x))
```

The positive solution is $\boxed{2}$.
"""
s, a = score_solution(test)
print(f"Score: {s:.1f}, Answer: {a}, Normalized: {normalize_answer(a)}")

print("\nNormalization regression tests:")
_tests = [
    # (raw_a, raw_b, should_match, label)
    (r'\frac{1}{3}',              r'\left(\frac{1}{3}\right)',      True,  "frac + \\left\\right"),
    (r'(\frac{1}{4})',            r'\frac{1}{4}',                   True,  "outer parens single value"),
    (r'\dfrac{10}{133}',          r'\frac{10}{133}',                True,  "dfrac vs frac"),
    (r'\frac{\pi-2}{2}',          r'\frac{\pi}{2}-1',               True,  "symbolic pi equivalence"),
    (r'(3\sqrt{7})',              r'3\sqrt{7}',                     True,  "outer parens sqrt"),
    (r'(36^\circ)',               r'36^\circ',                      True,  "outer parens circ"),
    (r'(\frac{1}{2})',            r'\frac{1}{2}',                   True,  "outer parens frac12"),
    (r'((1,5))',                  r'(1,5)',                          True,  "double outer parens tuple"),
    (r'(2,7),(3,17)',             r'((2,7))\,\text{and}\,((3,17))', True,  "tuple set with and"),
    (r'\frac{1}{3}',              r'\frac{1}{4}',                   False, "different fracs"),
    (r'(2,7)',                    r'(3,7)',                          False, "different tuples"),
]
all_ok = True
for raw_a, raw_b, expected, label in _tests:
    na, nb = normalize_answer(raw_a), normalize_answer(raw_b)
    match = answers_match(na, nb)
    status = "OK" if match == expected else "FAIL"
    if match != expected:
        all_ok = False
    print(f"  [{status}] {label}: '{na}' vs '{nb}' → {match}")

print(f"\n{'All tests passed!' if all_ok else 'Some tests FAILED — check above.'}")
print("Verification engine ready.")

---
## 3. Problem Solver — Best-of-N with Majority Vote

For each problem:
1. Generate N candidate solutions
2. Extract and normalize answers from each
3. Score each solution (code exec + answer presence)
4. Pick answer by **weighted majority vote** (votes weighted by solution score)

In [ ]:
from collections import Counter

SYSTEM_PROMPT = (
    "You are an expert math competition solver.\n\n"
    "For EVERY problem, follow this process:\n"
    "1. Carefully read and understand what is being asked.\n"
    "2. Plan your mathematical approach.\n"
    "3. Use Python (sympy / numpy) code blocks to carry out all computations "
    "   — do NOT do arithmetic in your head.\n"
    "4. After reaching a candidate answer, write a Python block that VERIFIES "
    "   it satisfies every condition in the problem.\n"
    "5. State your final, verified answer in \\boxed{}.\n\n"
    "Rules:\n"
    "- Always use sympy for exact algebra (solving equations, simplifying, "
    "  factoring, number-theory checks).\n"
    "- If a problem asks for ALL solutions, enumerate them systematically in "
    "  code — do not guess.\n"
    "- Never leave a numerical computation to mental arithmetic.\n"
    "- Your \\boxed{} answer must match what your Python code computed.\n"
)


def build_prompt(problem):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def solve_problem(problem, n=SOLUTIONS_PER_PROBLEM):
    """Generate n solutions, return (best_answer, details)."""
    prompt = build_prompt(problem)

    params = SamplingParams(
        temperature=0.7,
        top_p=0.95,
        max_tokens=MAX_TOKENS,
        n=n,
        stop=["<|im_end|>", "<|endoftext|>"],
    )

    outputs = llm.generate([prompt], params)[0]

    # Score and extract answers
    candidates = []
    for out in outputs.outputs:
        text = out.text
        sc, raw_ans = score_solution(text)
        norm_ans = normalize_answer(raw_ans)
        candidates.append({
            "text": text,
            "raw_answer": raw_ans,
            "norm_answer": norm_ans,
            "score": sc,
        })

    # Weighted majority vote
    vote_weights = {}
    for c in candidates:
        a = c["norm_answer"]
        if a:
            vote_weights[a] = vote_weights.get(a, 0) + max(c["score"], 0.1)

    if not vote_weights:
        return None, candidates

    best_answer = max(vote_weights, key=vote_weights.get)
    return best_answer, candidates


# Quick test
test_ans, test_cands = solve_problem("What is 2 + 2?")
print(f"Answer: {test_ans}")
print(f"Candidates: {len(test_cands)}")
ans_dist = Counter(c['norm_answer'] for c in test_cands if c['norm_answer'])
print(f"Answer distribution: {dict(ans_dist)}")

---
## 4. Load OpenMathReasoning TIR Dataset

Stream problems from `nvidia/OpenMathReasoning` (TIR split).  
Each row has: `problem`, `expected_answer`, `generated_solution`, `problem_source`, `pass_rate_72b_tir`.

We sample a benchmark set, stratified by difficulty (pass rate).

In [6]:
from datasets import load_dataset
import pandas as pd

# --- Config ---
EVAL_N = 100  # number of problems to benchmark
SEED = 42

print("Loading nvidia/OpenMathReasoning (tir split)...")
ds = load_dataset("nvidia/OpenMathReasoning", split="tir", streaming=True)

# Stream and collect problems (deduplicate by problem text)
seen_problems = set()
rows = []

for example in ds:
    prob_text = example["problem"].strip()
    if prob_text in seen_problems:
        continue
    seen_problems.add(prob_text)

    rows.append({
        "problem": prob_text,
        "expected_answer": str(example["expected_answer"]).strip(),
        "problem_source": example.get("problem_source", "unknown"),
        "pass_rate": example.get("pass_rate_72b_tir", "unknown"),
        "generated_solution": example["generated_solution"],
    })

    if len(rows) >= EVAL_N * 5:  # collect extra so we can sample
        break

all_problems_df = pd.DataFrame(rows)
print(f"Collected {len(all_problems_df)} unique problems")

# Sample benchmark set
eval_df = all_problems_df.sample(n=min(EVAL_N, len(all_problems_df)), random_state=SEED).reset_index(drop=True)

print(f"\nBenchmark set: {len(eval_df)} problems")
print(f"\nProblem sources:")
print(eval_df["problem_source"].value_counts().head(10).to_string())
print(f"\nPass rate distribution:")
print(eval_df["pass_rate"].value_counts().head(10).to_string())
print(f"\nSample problem:")
print(f"  {eval_df.iloc[0]['problem'][:200]}...")
print(f"  Expected: {eval_df.iloc[0]['expected_answer']}")

Loading nvidia/OpenMathReasoning (tir split)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Collected 500 unique problems

Benchmark set: 100 problems

Problem sources:
problem_source
aops_c6_high_school_olympiads    54
aops_c4_high_school_math         34
aops_c7_college_math              8
aops_c5_contests_amp_programs     4

Pass rate distribution:
pass_rate
0.96875    11
0.90625    11
0.0         7
0.59375     6
0.875       6
0.9375      4
0.75        4
0.71875     4
0.65625     4
0.46875     3

Sample problem:
  Solve the equation $x^3 - y^3 = x^2 + 2x + y^2$ for integers $x$ and $y$....
  Expected: \((x,y) \in \{(3,2), (-1,-1), (-1,0), (0,-1), (0,0), (2,-1), (2,0)\}\)


In [7]:
import time
import json
from collections import Counter
from tqdm.auto import tqdm

n_problems = len(eval_df)
print(f"Solving {n_problems} problems, {SOLUTIONS_PER_PROBLEM} solutions each...")
print(f"Total generations: {n_problems * SOLUTIONS_PER_PROBLEM}")
print()

results = []
t0 = time.time()

for i, row in eval_df.iterrows():
    answer, candidates = solve_problem(row["problem"])
    expected = normalize_answer(row["expected_answer"])
    correct = answers_match(answer or "", expected)

    # Check coverage
    any_correct = any(
        answers_match(c["norm_answer"], expected)
        for c in candidates if c["norm_answer"]
    )

    # Answer distribution
    ans_counts = Counter(c["norm_answer"] for c in candidates if c["norm_answer"])

    results.append({
        "problem": row["problem"][:80],
        "expected": expected,
        "predicted": answer,
        "correct": correct,
        "any_correct": any_correct,
        "source": row["problem_source"],
        "pass_rate": row["pass_rate"],
        "n_answers": len(ans_counts),
        "top_answer_votes": ans_counts.most_common(1)[0][1] if ans_counts else 0,
        "answer_dist": dict(ans_counts.most_common(5)),
    })

    status = "CORRECT" if correct else ("COVERED" if any_correct else "MISSED")
    print(f"  [{i+1:>3}] {status:<8} expected={expected:<12} got={answer or 'None':<12} "
          f"dist={dict(ans_counts.most_common(3))}")

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")

Solving 100 problems, 16 solutions each...
Total generations: 1600



Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  [  1] MISSED   expected=((x,y)in(3,2),(-1,-1),(-1,0),(0,-1),(0,0),(2,-1),(2,0)) got=(0,0),(0,-1),(-1,0),(-1,-1),(2,0),(2,-1) dist={'(0,0),(0,-1),(-1,0),(-1,-1),(2,0),(2,-1)': 4, '(3,2),(0,-1),(-1,0),(0,0),(-1,-1)': 2, '(0,0),(-1,-1),(3,2),(0,-1),(-1,0)': 2}


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  [ 99] MISSED   expected=(left(frac1-sqrt22,frac1+sqrt22right)) got=frac1-sqrt22andfrac1+sqrt22 dist={'frac1-sqrt22andfrac1+sqrt22': 3, 'frac1+sqrt22': 3, 'left(frac12-fracsqrt22,frac12+fracsqrt22right)': 1}


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  [100] MISSED   expected=(fracxsinx-cosxxcosx+sinx+c) got=fracxxcosx+sinx+c dist={'fracxxcosx+sinx+c': 3, '-frac1xcosx+sinx+c': 3, '-fracxxcosx+sinx+c': 2}

Done in 1826.9s


In [ ]:
results_df = pd.DataFrame(results)
n = len(results_df)
accuracy = results_df["correct"].mean()
coverage = results_df["any_correct"].mean()

print("=" * 70)
print(f"RESULTS — {n} problems × {SOLUTIONS_PER_PROBLEM} solutions")
print(f"Dataset: nvidia/OpenMathReasoning (TIR)")
print(f"Model:   {MODEL_ID}")
print("=" * 70)
print(f"  Accuracy (majority vote):  {accuracy:.1%} ({results_df['correct'].sum()}/{n})")
print(f"  Coverage (any correct):    {coverage:.1%} ({results_df['any_correct'].sum()}/{n})")
print(f"  Gap (room to improve):     {coverage - accuracy:.1%}")
print(f"  Avg distinct answers:      {results_df['n_answers'].mean():.1f}")
print(f"  Avg top answer votes:      {results_df['top_answer_votes'].mean():.1f}/{SOLUTIONS_PER_PROBLEM}")
print(f"  Time:                      {elapsed:.1f}s")

# By problem source
print(f"\n{'='*70}")
print("BY PROBLEM SOURCE")
print("=" * 70)
for src in results_df["source"].unique():
    src_data = results_df[results_df["source"] == src]
    if len(src_data) < 2:
        continue
    s_acc = src_data["correct"].mean()
    s_cov = src_data["any_correct"].mean()
    print(f"  {src:<40} n={len(src_data):>3}  acc={s_acc:.0%}  cov={s_cov:.0%}")

# By pass rate (difficulty proxy)
print(f"\n{'='*70}")
print("BY DIFFICULTY (pass_rate_72b_tir)")
print("=" * 70)
for pr in sorted(results_df["pass_rate"].unique()):
    pr_data = results_df[results_df["pass_rate"] == pr]
    if len(pr_data) < 2:
        continue
    p_acc = pr_data["correct"].mean()
    p_cov = pr_data["any_correct"].mean()
    print(f"  pass_rate={str(pr):<12} n={len(pr_data):>3}  acc={p_acc:.0%}  cov={p_cov:.0%}")

# Problem details
print(f"\n{'='*70}")
print("PROBLEM DETAILS")
print("=" * 70)
for _, r in results_df.iterrows():
    mark = "OK" if r["correct"] else ("--" if r["any_correct"] else "XX")
    print(f"  [{mark}] exp={r['expected']:<12} got={str(r['predicted']):<12} {r['problem'][:50]}")

# Save
summary = {
    "model": MODEL_ID,
    "dataset": "nvidia/OpenMathReasoning (tir)",
    "n_problems": n,
    "solutions_per": SOLUTIONS_PER_PROBLEM,
    "accuracy": round(accuracy, 4),
    "coverage": round(coverage, 4),
    "time_s": round(elapsed, 1),
}
with open("results/baseline.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved to results/baseline.json")
print(f"\nBASELINE: {accuracy:.1%} accuracy. Beam search must beat this.")

---
## 5. Beam Search with Verification

Instead of generating complete solutions and hoping one is right,  
generate **step by step** and prune bad paths early.

Algorithm:
1. Start with the problem as the root
2. Generate `beam_width` candidate next-steps
3. Score each step (does the code run? is reasoning consistent?)
4. Keep top-k steps, discard the rest
5. Repeat until a `\boxed{}` answer appears
6. Majority vote across completed beams

In [ ]:
def generate_continuations(partial_solution, problem, n_candidates=4):
    """Generate n candidate next-steps for a partial solution."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    if partial_solution:
        # Continue from where we left off
        messages.append({"role": "assistant", "content": partial_solution})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=(not partial_solution)
    )
    # If continuing, don't add generation prompt — we're mid-response
    if partial_solution:
        # Remove trailing special tokens to continue generation
        prompt = prompt.rstrip()
        if prompt.endswith("<|im_end|>"):
            prompt = prompt[:-len("<|im_end|>")].rstrip()

    params = SamplingParams(
        temperature=0.8,
        top_p=0.95,
        max_tokens=512,  # one step at a time
        n=n_candidates,
        stop=["<|im_end|>", "<|endoftext|>"],
    )

    outputs = llm.generate([prompt], params)[0]
    return [o.text for o in outputs.outputs]


def score_step(full_solution_so_far):
    """Score a partial solution. Rewards progress, penalizes errors."""
    score = 0.0

    # Has answer? Big bonus — means we reached a conclusion
    if extract_answer(full_solution_so_far):
        score += 3.0

    # Code execution check
    passed, failed, output = execute_code_blocks(full_solution_so_far)
    if passed > 0:
        score += 1.0 * passed
    if failed > 0:
        score -= 2.0 * failed  # heavily penalize broken code

    # Length penalty — prefer concise solutions
    score -= 0.001 * len(full_solution_so_far)

    return score


def beam_search_solve(problem, beam_width=4, max_steps=5):
    """Solve a problem using beam search over solution steps."""
    # Each beam is (partial_solution_text, cumulative_score)
    beams = [("", 0.0)]
    completed = []  # beams that produced a \boxed{} answer

    for step in range(max_steps):
        all_candidates = []

        for partial, cum_score in beams:
            continuations = generate_continuations(partial, problem, n_candidates=beam_width)

            for cont in continuations:
                full = partial + cont
                step_score = score_step(full)
                total_score = cum_score + step_score
                all_candidates.append((full, total_score))

                # Check if this beam is done
                if extract_answer(full):
                    completed.append((full, total_score))

        # Keep top beam_width candidates (that don't have answers yet)
        not_done = [(s, sc) for s, sc in all_candidates if not extract_answer(s)]
        not_done.sort(key=lambda x: x[1], reverse=True)
        beams = not_done[:beam_width]

        if not beams and completed:
            break  # all beams finished

    # Also add remaining beams as completed (even without boxed answer)
    completed.extend(beams)

    if not completed:
        return None, []

    # Weighted majority vote across completed beams
    vote_weights = {}
    for sol, sc in completed:
        ans = normalize_answer(extract_answer(sol))
        if ans:
            vote_weights[ans] = vote_weights.get(ans, 0) + max(sc, 0.1)

    if not vote_weights:
        return None, completed

    best = max(vote_weights, key=vote_weights.get)
    return best, completed


# Quick test
bs_ans, bs_beams = beam_search_solve("What is 2 + 2?", beam_width=3, max_steps=3)
print(f"Beam search answer: {bs_ans}")
print(f"Completed beams: {len(bs_beams)}")

In [ ]:
print(f"Beam search benchmark on {len(eval_df)} problems...")
print(f"beam_width=4, max_steps=5")
print()

beam_results = []
t0 = time.time()

for i, row in eval_df.iterrows():
    answer, beams = beam_search_solve(row["problem"], beam_width=4, max_steps=5)
    expected = normalize_answer(row["expected_answer"])
    correct = answers_match(answer or "", expected)

    # Check if any beam got it right
    any_correct = any(
        answers_match(normalize_answer(extract_answer(s)) or "", expected)
        for s, _ in beams
    )

    beam_results.append({
        "problem": row["problem"][:80],
        "expected": expected,
        "predicted": answer,
        "correct": correct,
        "any_correct": any_correct,
        "source": row["problem_source"],
        "n_beams": len(beams),
    })

    status = "CORRECT" if correct else ("COVERED" if any_correct else "MISSED")
    print(f"  [{i+1:>3}] {status:<8} expected={expected:<12} got={answer or 'None':<12}")

beam_elapsed = time.time() - t0

beam_df = pd.DataFrame(beam_results)
beam_acc = beam_df["correct"].mean()
beam_cov = beam_df["any_correct"].mean()

print(f"\n{'='*70}")
print(f"BEAM SEARCH vs BEST-OF-N")
print(f"{'='*70}")
print(f"  {'Method':<25} {'Accuracy':<15} {'Coverage':<15} {'Time':<10}")
print(f"  {'-'*65}")
print(f"  {'Best-of-'+str(SOLUTIONS_PER_PROBLEM):<25} {accuracy:<15.1%} {coverage:<15.1%} {elapsed:<10.1f}s")
print(f"  {'Beam (w=4, s=5)':<25} {beam_acc:<15.1%} {beam_cov:<15.1%} {beam_elapsed:<10.1f}s")
delta = beam_acc - accuracy
print(f"\n  Delta: {'+' if delta >= 0 else ''}{delta:.1%}")

# Save
beam_summary = {
    "model": MODEL_ID,
    "dataset": "nvidia/OpenMathReasoning (tir)",
    "method": "beam_search",
    "beam_width": 4,
    "max_steps": 5,
    "accuracy": round(beam_acc, 4),
    "coverage": round(beam_cov, 4),
    "time_s": round(beam_elapsed, 1),
    "baseline_accuracy": round(accuracy, 4),
}
with open("results/beam_search.json", "w") as f:
    json.dump(beam_summary, f, indent=2)
print(f"\nSaved to results/beam_search.json")

---
## Next Steps

Based on results above:
- If **coverage is high but accuracy is low** → improve ranking/voting
- If **coverage is low** → need better generation (fine-tune model)
- If **beam search helps** → invest in MCTS next
- If **beam search doesn't help** → focus on best-of-N with more N + better verification